In [ ]:
import os

import chemiscope
import ipi
from ase.visualize import view
import nqetools as nqe
# This follows:
# https://atomistic-cookbook.org/examples/pi-metad/pi-metad.html

In [ ]:
# Make a directory to store everything
directory_opti = "opti"
directory_md = "md"
directory_meta_md = "meta_md"
directory_meta_pimd = "meta_pimd"

n_beads = 8
timestep = 1.0  # fs
total_steps = 10000
total_steps_md = 100
stride = 10
temperature = 300
thermostat = 'smart_sampling_1ps_n6_w2'
md_type = "NVT-GLE"
driver_code = 'zundel'

# Plumed hills settings
n_bins = 100
stride_hills = 100
cv_limits = [[0.21, 0.31], [-1, 1]]

# Plumed settings
plumed_type_mtd = "mtd-coord"
plumed_args_mtd = {'sigma': [0.005, 0.05],
                   'bias': 10,
                   'height': 0.04, }

plumed_type_opes = "opes-coord"
plumed_args_opes = {'barrier': 0.08,
                    'stride_hills': stride_hills,
                    'explore': False,}




In [ ]:
atoms = nqe.read_ipi_xyz("h5o2+.xyz")[-1]
atoms.center()
# view(atoms)

In [ ]:
# Run minimization
output = nqe.run_optimise(directory_opti,
                          atoms,
                          driver=driver_code)
atoms_opti, output_data_opti, output_desc_opti = output

In [ ]:
# Plot the energy of the minimisation
nqe.plot_step_energy(output_data_opti)

In [ ]:
# Run unbiased MD
output = nqe.run_md(directory_md,
                    atoms_opti,
                    driver=driver_code,
                    total_steps=total_steps_md,
                    temperature=temperature,
                    timestep=timestep,
                    thermostat=thermostat,
                    md_type=md_type,
                    stride=stride,
                    n_beads=1)
atoms_md, output_data_md, output_desc_md = output

In [ ]:
idx1 = 0
idx2 = 1
# Get a list of all the atom indexes
tmp = list(range(len(atoms)))
print(tmp)

# remove idx1 and idx2
tmp.remove(idx1)
tmp.remove(idx2)
print(tmp)



# add one to all the indexes
tmp = [x + 1 for x in tmp]
print(tmp)

group_b_str=",".join([str(x) for x in tmp])
print(group_b_str)


In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_md)

In [ ]:
# Run metadynamics
output = nqe.run_plumed_md(directory_meta_md,
                           atoms_md,
                           driver=driver_code,
                           total_steps=total_steps,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           stride=stride,
                           n_beads=1,
                           plumed_type=plumed_type_mtd,
                           plumed_args=plumed_args_mtd)
atoms_meta_md, output_data_meta_md, output_desc_meta_md = output

In [ ]:
# colvar_data = ipi.read_trajectory(os.path.join(directory_meta_md, "md.colvar_0"), format="extras")[
#     "d,c1.lessthan,c2.lessthan,dc,mtd.bias"
# ]
# traj_data = ipi.read_trajectory(os.path.join(directory_meta_md, "md.pos_0.xyz"))
# # Chemiscope plot
# chemiscope.show(
#     frames=traj_data,
#     properties=dict(
#         d_OO=10 * colvar_data[:, 0],  # nm to Å
#         delta_coord=colvar_data[:, 1],
#         bias=27.211386 * output_data["ensemble_bias"],  # Ha to eV
#         time=2.4188843e-05 * output_data["time"],  # atomictime to ps
#     ),  # attime to ps
#     settings=chemiscope.quick_settings(
#         x="d_OO", y="delta_coord", z="bias", color="time", trajectory=True
#     ),
#     mode="default",
# )

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_md)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_md)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_md)

In [ ]:
# Run the hills command
nqe.run_plumed_hills(directory_meta_md,
                     temperature=temperature,
                     bins=n_bins,
                     stride=stride_hills,
                     cv=cv_limits)
# Load the free energy surface data
fes_arrays_meta_md = nqe.load_fes_data(directory_meta_md, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps, fes_arrays_meta_md)
# Plot the free energy surface convergence
nqe.plot_fes_contourf_series(fes_arrays_meta_md, fes_times)

In [ ]:
# Run PIMD metadynamics
output = nqe.run_plumed_md(directory_meta_pimd,
                           atoms_md,
                           driver=driver_code,
                           total_steps=total_steps,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           stride=stride,
                           n_beads=n_beads,
                           plumed_type=plumed_type_mtd,
                           plumed_args=plumed_args_mtd)
atoms_meta_pimd, output_data_meta_pimd, output_desc_meta_pimd = output

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_pimd)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_pimd)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_pimd)

In [ ]:
# Run the hills command
nqe.run_plumed_hills(directory_meta_pimd,
                     temperature=temperature,
                     bins=n_bins,
                     stride=stride_hills,
                     cv=cv_limits)
# Load the free energy surface data
fes_arrays_meta_pimd = nqe.load_fes_data(directory_meta_pimd, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps, fes_arrays_meta_pimd)
# Plot the free energy surface convergence
nqe.plot_fes_contourf_series(fes_arrays_meta_pimd, fes_times)

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_contour_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_contourf_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
nqe.plot_fes_sep(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
# Run OPES metadynamics
output = nqe.run_plumed_md(directory_meta_md,
                           atoms_md,
                           driver=driver_code,
                           total_steps=total_steps,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           stride=stride,
                           n_beads=1,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_md, output_data_meta_md, output_desc_meta_md = output

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_md)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_md)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_md)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_md,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)

fes_arrays_meta_md = nqe.load_fes_data(directory_meta_md, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps, fes_arrays_meta_md)
# Plot the free energy surface convergence
nqe.plot_fes_contourf_series(fes_arrays_meta_md, fes_times)

In [ ]:
# Run PIMD OPES metadynamics
output = nqe.run_plumed_md(directory_meta_pimd,
                           atoms_md,
                           driver=driver_code,
                           total_steps=total_steps,
                           temperature=temperature,
                           timestep=timestep,
                           thermostat=thermostat,
                           md_type=md_type,
                           stride=stride,
                           n_beads=n_beads,
                           plumed_type=plumed_type_opes,
                           plumed_args=plumed_args_opes)
atoms_meta_pimd, output_data_meta_pimd, output_desc_meta_pimd = output

In [ ]:
# Plot the time evolution of the energy and bias
nqe.plot_time_potential_bias(output_data_meta_pimd)

In [ ]:
# Plot the energy conservation
nqe.plot_time_energy_conservation(output_data_meta_pimd)

In [ ]:
# Plot the time evolution of the temperature
nqe.plot_time_temperature(output_data_meta_pimd)

In [ ]:
# Run the hills command
nqe.run_plumed_hills_opes(directory_meta_pimd,
                          temperature=temperature,
                          bins=n_bins,
                          cv=cv_limits)

fes_arrays_meta_pimd = nqe.load_fes_data(directory_meta_pimd, n_bins)
fes_times = nqe.get_fes_times(timestep, total_steps, fes_arrays_meta_pimd)
# Plot the free energy surface convergence
nqe.plot_fes_contourf_series(fes_arrays_meta_pimd, fes_times)

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_contour_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
# Plot the comparison between MD and PIMD, compare the last frame
nqe.plot_fes_contourf_compare(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])

In [ ]:
nqe.plot_fes_sep(fes_arrays_meta_md[-1], fes_arrays_meta_pimd[-1])